# Choosing the example regions: the 2 x 3 block across all $\psi$

`region_space.ipynb` illustrates the two knobs with one 2 x 3 block -- three regions on top
as neuron clouds, the same three below as population manifolds. Which $\psi$ contrast to use
was picked by hand. This notebook draws that block for every $\psi$, so it can be picked on
the evidence instead.

In each block the right-hand pair is **fixed**: the reference region ($a$ = 0) and the same
region made categorical ($a$ = 0.95), both at $\psi_{\text{ref}}$. Only the **left** panel
moves, sweeping $\psi$. So each block shows one candidate contrast, and the number above it
is the Procrustes distance between the swept region and the reference.

### Why 8 values over 180 deg and not 360

$\psi$ is the *orientation* of a region's long axis in skew-width space, and the cloud is
symmetric about its own centre, so $\psi$ and $\psi + 180^\circ$ generate the same region.
Measured directly: $d$(30, 210) = 2.09 and $d$(75, 255) = 1.17 -- sampling noise -- against
$d$(30, 75) = 12.74 for a genuinely different orientation. A 360 deg sweep would therefore
show four distinct examples twice each. 180 deg covers the whole space.

**All blocks share one frame.** The clouds use a single PCA fitted to every region at once,
and every manifold is rotated into the *same* reference frame, so blocks are comparable with
each other and not just internally.

In [ ]:
from pathlib import Path

from shapemetrics import paths
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.patches import Ellipse
from sklearn.decomposition import PCA

# Anchor on the directory holding the simulation modules, so the notebook runs
# whether the kernel starts in clustering-simulation or at the repo root.
# Path.cwd() alone fails with ModuleNotFoundError from anywhere else.
R = paths.figure_code("Figure1").region_space
import shapemetrics as sm

OUT = paths.set_figure("Figure1")
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

N_SWEEP = 8
PSI_DEG = np.linspace(0, 180, N_SWEEP, endpoint=False)     # the whole space
PSI_REF = -R.PSI                                           # the fixed right-hand pair
SEED = 1

# regions 0..7 sweep psi at a = 0; region 8 is the reference; region 9 is the
# reference made categorical.  Simulated in ONE call so a single PCA frame
# serves every block.
amounts = [0.0] * N_SWEEP + [0.0, 0.95]
psis = list(np.radians(PSI_DEG)) + [PSI_REF, PSI_REF]
X, reg = R.simulate(amounts, psis, seed=SEED)
I_REF, I_CAT = N_SWEEP, N_SWEEP + 1

print(f"{len(np.unique(reg))} regions x {R.N_NEURONS} neurons, "
      f"psi_ref = {np.degrees(PSI_REF):+.0f} deg")

## How different is each swept region from the reference?

In [ ]:
D = sm.distance_matrix(X, reg, n_pcs=R.N_PCS, alpha=1)
d_ref = D[:N_SWEEP, I_REF]                 # swept region vs the reference
d_cat = D[I_REF, I_CAT]                    # the a-only contrast, for scale

print(f"the a-only contrast (a = 0 vs 0.95 at fixed psi):  d = {d_cat:.2f}\n")
print(f"{'psi':>7} {'d to reference':>15} {'ratio to the a-only contrast':>30}")
for k in np.argsort(-d_ref):
    print(f"{PSI_DEG[k]:>6.1f} {d_ref[k]:>15.2f} {d_ref[k] / d_cat:>30.1f}x")

A good example maximises that ratio: the $\psi$ contrast should dwarf the $a$ contrast, since
the figure's whole claim is that one moves the geometry and the other does not.

## The blocks

In [ ]:
# ---- one shared frame for the clouds -------------------------------------
We = PCA(2).fit_transform(X)
_A = np.c_[We, np.ones(len(We))]
_S = np.c_[((X - R.MEAN) @ R.MODES.T / R.SCALE)[:, 0],
           -((X - R.MEAN) @ R.MODES.T / R.SCALE)[:, 1]]
_B = np.linalg.lstsq(_A, _S, rcond=None)[0][:2]
SHAPE_DIRS = _B / np.linalg.norm(_B, axis=0)          # columns: skew, width

cl = {i: We[reg == i] - We[reg == i].mean(0) for i in np.unique(reg)}
elim = 1.20 * np.percentile(np.abs(np.concatenate(list(cl.values()))), 99.5)


def align_to(A, B):
    # rotate/reflect B onto A -- the transform the shape metric optimises over
    U, _, Vt = np.linalg.svd(B.T @ A)
    return B @ (U @ Vt)


# ---- one shared frame for the manifolds ----------------------------------
MAN = {}
for i in np.unique(reg):
    m = PCA(2).fit_transform(X[reg == i].T)
    m = m - m.mean(0)
    MAN[i] = m / np.linalg.norm(m)
MREF = MAN[I_REF]
MAN = {i: align_to(MREF, m) for i, m in MAN.items()}     # all into ONE frame
# PC 1 carries ~97% of the manifold's variance and PC 2 ~2%, so equal limits
# draw it as a sliver.  Each axis gets its own limit, which stretches PC 2 to
# fill the panel.  The display is then anisotropic and no longer metric -- but
# the SAME stretch is applied to every panel, and the alignment above was
# computed in the true isotropic space, so grey-vs-colour stays exact.
_allm = np.concatenate(list(MAN.values()))
mlx = 1.06 * np.abs(_allm[:, 0]).max()
mly = 1.06 * np.abs(_allm[:, 1]).max()
print(f"PC1 : PC2 extent = {mlx / mly:.1f} : 1  (stretch applied to every panel)")


def gauss_ellipse(ax, P, nsd=2.0):
    w, V = np.linalg.eigh(np.cov(P, rowvar=False))
    ax.add_patch(Ellipse(P.mean(0), 2 * nsd * np.sqrt(w[-1]),
                         2 * nsd * np.sqrt(w[-2]),
                         angle=np.degrees(np.arctan2(V[1, -1], V[0, -1])),
                         fill=False, lw=0.8, ec="black", zorder=3))


def draw_cloud(ax, i, split=False):
    P = cl[i]
    u = np.linalg.eigh(np.cov(P, rowvar=False))[1][:, -1]
    for j, nm in enumerate(("skew", "width")):
        d = SHAPE_DIRS[:, j]
        ax.plot([-elim * d[0], elim * d[0]], [-elim * d[1], elim * d[1]],
                color="0.88", lw=0.6, zorder=0)
    ax.plot([-elim * u[0], elim * u[0]], [-elim * u[1], elim * u[1]],
            color="0.65", lw=0.8, ls="--", zorder=1)
    ax.scatter(P[:, 0], P[:, 1], s=4, facecolor="0.6", edgecolor="black",
               linewidths=0.2, alpha=0.9, zorder=2)
    if split:
        pr = P @ u
        for m in (pr < 0, pr >= 0):
            gauss_ellipse(ax, P[m])
    else:
        gauss_ellipse(ax, P)
    ax.set(xlim=(-elim, elim), ylim=(-elim, elim), xticks=[], yticks=[])
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True, bottom=True)


def draw_manifold(ax, i):
    ax.plot(MREF[:, 0], MREF[:, 1], color="0.78", lw=1.8, zorder=0)
    mx, my = MAN[i][::2].T
    # bead chain as in panel b of figure 1; these panels are smaller than the
    # summary figure's, so s scales down with panel area
    ax.scatter(mx, my, lw=0, s=22, alpha=1, zorder=2,
               c=np.linspace(0, 1, len(mx)),
               cmap=ListedColormap(sns.color_palette("husl", len(mx))))
    ax.set(xlim=(-mlx, mlx), ylim=(-mly, mly), xticks=[], yticks=[])
    ax.set_box_aspect(1); sns.despine(ax=ax, left=True, bottom=True)


# ---- 8 blocks, arranged 4 block-rows x 2 block-columns --------------------
PANEL = 1.45
fig, axes = plt.subplots(8, 6, figsize=(6 * PANEL, 8 * PANEL))
for b in range(N_SWEEP):
    br, bc = b // 2, b % 2
    r0, c0 = 2 * br, 3 * bc
    for k, (i, split) in enumerate(((b, False), (I_REF, False), (I_CAT, True))):
        draw_cloud(axes[r0, c0 + k], i, split=split)
        draw_manifold(axes[r0 + 1, c0 + k], i)
    axes[r0, c0].set_title(
        f"$\\psi$ = {PSI_DEG[b]:.0f}$^\\circ$    $d$ = {d_ref[b]:.1f}"
        f"  ({d_ref[b] / d_cat:.1f}x)", fontsize=8, pad=4, loc="left")
    for k, lab in enumerate(("swept", "reference", "reference, $a$ = 0.95")):
        axes[r0, c0 + k].set_xlabel(lab, fontsize=6, labelpad=2, color="0.35")

fig.tight_layout(w_pad=0.8, h_pad=1.4)
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"psi_sweep.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## Reading it

Each block is a candidate for the figure in `region_space.ipynb`. Two things make a good one:

1. **The left manifold should sit clearly off the grey reference** while the right manifold
   sits on it -- that is the whole dissociation, visible in one row.
2. **The left cloud should look obviously different from the middle one**, so the neuron-level
   panel and the manifold panel tell the same story about $\psi$.

The distance ratio in each title is the quantitative version of (1): how many times larger
the $\psi$ contrast is than the $a$ contrast. The $\psi$ values near the reference are poor
examples -- the manifolds nearly coincide -- and the ratio makes that explicit rather than a
matter of taste.